In [1]:
import os
import sys
from dask.distributed import Client
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json', threads_per_worker=2, n_workers=6)
client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler.json')  
# client = Client(scheduler_file='/proj/kimyy/Dropbox/source/python/all/mpi/scheduler_10.json')  

# add private module path for workers
# client.run(lambda: os.environ.update({'PYTHONPATH': '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'}))
# def add_path():
#     if '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2' not in sys.path:
#         sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')

# client.run(add_path)

def setup_module_path():
    module_path = '/proj/kimyy/Dropbox/source/python/all/Modules/CESM2'
    if module_path not in sys.path:
        sys.path.append(module_path)

client.run(setup_module_path)

client

<Client: 'tcp://203.247.189.224:45831' processes=5 threads=90, memory=419.10 GiB>

In [2]:
# load public modules

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
import cmocean
from cmcrameri import cm
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import cftime
import pop_tools
from pprint import pprint
import time
import subprocess
import re as re_mod
import cftime
import datetime
from scipy.stats import ttest_1samp
import xcesm

In [3]:
# load private modules

import sys
sys.path.append('/proj/kimyy/Dropbox/source/python/all/Modules/CESM2')
from KYY_CESM2_preprocessing import CESM2_config
# import KYY_CESM2_preprocessing
# import importlib
# importlib.reload(KYY_CESM2_preprocessing)

In [4]:
lim_varnames_2d = [
    "TAUX",
    "TAUY",
    "HBLT",
    "SSH",
    "photoC_TOT_zint_100m"
]
lim_varnames_3d = [
    "TEMP",
    "SALT",
    "Fe",
    "NO3",
    "PO4",
    "SiO3",
    "VVEL",
]

lim_varnames_3d_w = [
    "QSW_3D",
]

lim_varnames_3d_150m = [
    "photoC_TOT",
    "PAR_avg",
]

cfgs = {}

for varname in lim_varnames_2d:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d_w:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

for varname in lim_varnames_3d_150m:
    cfg = CESM2_config()
    cfg.year_s = 1998
    cfg.year_e = 2007
    cfg.setvar(varname)
    cfgs[varname] = cfg

if cfgs[varname].comp=='ocn':
    ds_grid = pop_tools.get_grid('POP_gx1v7')



In [5]:
# define preprocessing function

# ds_grid_sub = ds_grid.where(mask, drop=True)
# area = ds_grid_sub['TAREA']
# weighted_avg_manual = xr.Dataset()

exceptcv = [
    'time', 'lon', 'lat', 'lev', 'TLONG', 'TLAT', 'z_w', 'z_w_top', 'z_t', 'ULONG', 'ULAT', 'VLONG', 'VLAT'
] + [cfg.var for cfg in cfgs.values()]

def process_coords_2d(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_t=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d_w(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_w_top=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)

def process_coords_3d_150m(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_t_150m=slice(0, 15))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)


def process_coords_3d_150m_LE(varname, ds, sd, ed, drop=True, except_coord_vars=exceptcv):
    import xcesm
    """Preprocessor function to drop all non-dim coords, which slows down concatenation."""
    coord_vars = []
    for v in np.array(ds.coords) :
        if not v in except_coord_vars:
            coord_vars += [v]
    for v in np.array(ds.data_vars) :
        if not v in except_coord_vars:
            coord_vars += [v]

    # ds = ds.where(mask, drop=True)
    ds= ds.isel(z_t_150m=slice(0, 1))
    
    if drop:
        ds= ds.drop(coord_vars)
        ds= ds.sel(time=slice(sd, ed))
        ds_rgd=ds[varname].utils.regrid()
        return ds_rgd
    else:
        return ds.set_coords(coord_vars)


start_date = cftime.DatetimeNoLeap(cfgs[varname].year_s, 2, 1)
end_date = cftime.DatetimeNoLeap(cfgs[varname].year_e+1, 1, 1)

In [6]:
def get_preprocess_func(varname, sd, ed):
    if varname in lim_varnames_2d:
        return lambda ds: process_coords_2d(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d:
        return lambda ds: process_coords_3d(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d_w:
        return lambda ds: process_coords_3d_w(
            varname, ds, sd, ed
        )

    elif varname in lim_varnames_3d_150m:
        return lambda ds: process_coords_3d_150m(
            varname, ds, sd, ed
        )

    else:
        raise ValueError(f"[ERROR] Unknown varname: {varname}")

In [7]:
# Read A_F_REF1 dataset


import datetime
import numpy as np
import xarray as xr

start_time = time.time()

for varname, cfg in cfgs.items():
    
    # variable name
    dname = cfg.var

    preprocess_func = get_preprocess_func(
        dname, start_date, end_date
    )
    
# --------------------------------------------------
# load A_F_REF1 file paths
# --------------------------------------------------
    cfg.A_F_REF1_path_load(dname) 

    cfg.A_F_REF1_ds = xr.open_mfdataset(
        cfg.A_F_REF1_file_list[0],
        combine="nested",
        concat_dim=[[*cfg.A_F_REF1_ensembles], "time"],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.isel(concat_dim=0)
    new_time = cfg.A_F_REF1_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_REF1_ds.sizes["time"]
    )
    cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.assign_coords(time=new_time)
    # cfg.A_F_REF1_ds = cfg.A_F_REF1_ds.compute()

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_REF1, ' + dname + ': ' + str(elapsed_time))


#------------------------------------------------------------------------------------------
    # load A_F_REF2 file paths
    cfg.A_F_REF2_path_load(dname)

    cfg.A_F_REF2_ds = xr.open_mfdataset(
        cfg.A_F_REF2_file_list[0],
        combine='nested',
        concat_dim=[[*cfg.A_F_REF2_ensembles], 'time'],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.isel(concat_dim=0)
    new_time = cfg.A_F_REF2_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_REF2_ds.sizes["time"]
    )
    cfg.A_F_REF2_ds = cfg.A_F_REF2_ds.assign_coords(time=new_time)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_REF2, ' + dname + ': ' + str(elapsed_time))


#------------------------------------------------------------------------------------------
    # load A_F_WIND file paths
    cfg.A_F_WIND_path_load(dname)

    cfg.A_F_WIND_ds = xr.open_mfdataset(
        cfg.A_F_WIND_file_list[0],
        combine='nested',
        concat_dim=[[*cfg.A_F_WIND_ensembles], 'time'],
        preprocess=preprocess_func,
        parallel=True,
    )

    # # rename ensemble dimension
    cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.isel(concat_dim=0)
    new_time = cfg.A_F_WIND_ds.time - np.array(
        [datetime.timedelta(days=15)] * cfg.A_F_WIND_ds.sizes["time"]
    )
    cfg.A_F_WIND_ds = cfg.A_F_WIND_ds.assign_coords(time=new_time)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print('elasped time for reading A_F_WIND, ' + dname + ': ' + str(elapsed_time))







elasped time for reading A_F_REF1, TAUX: 3.645519733428955
elasped time for reading A_F_REF2, TAUX: 5.1864588260650635
elasped time for reading A_F_WIND, TAUX: 6.79541277885437
elasped time for reading A_F_REF1, TAUY: 8.351992130279541
elasped time for reading A_F_REF2, TAUY: 9.263782262802124
elasped time for reading A_F_WIND, TAUY: 10.122346878051758
elasped time for reading A_F_REF1, HBLT: 11.058831930160522
elasped time for reading A_F_REF2, HBLT: 11.917545795440674
elasped time for reading A_F_WIND, HBLT: 12.770023345947266
elasped time for reading A_F_REF1, SSH: 13.697405099868774
elasped time for reading A_F_REF2, SSH: 14.688049077987671
elasped time for reading A_F_WIND, SSH: 15.616741418838501
elasped time for reading A_F_REF1, photoC_TOT_zint_100m: 16.50412631034851
elasped time for reading A_F_REF2, photoC_TOT_zint_100m: 17.358193159103394


ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'h5netcdf', 'scipy']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
#LE read

cfgs['photoC_TOT'].LE_path_load('photoC_TOT') 

cfgs['photoC_TOT'].LE_ds = xr.open_mfdataset(
    cfgs['photoC_TOT'].LE_file_list[0],
    combine="nested",
    concat_dim=[[*cfgs['photoC_TOT'].LE_ensembles], "time"],
    preprocess=lambda ds: process_coords_3d_150m_LE(
            'photoC_TOT', ds, sd, ed
        ),
    parallel=True,
)

# # rename ensemble dimension
cfgs['photoC_TOT'].LE_ds = cfgs['photoC_TOT'].LE_ds.isel(concat_dim=0)
new_time = cfgs['photoC_TOT'].LE_ds.time - np.array(
    [datetime.timedelta(days=15)] * cfgs['photoC_TOT'].LE_ds.sizes["time"]
)
cfgs['photoC_TOT'].LE_ds = cfgs['photoC_TOT'].LE_ds.assign_coords(time=new_time)

end_time = time.time()
elapsed_time = end_time - start_time
print('elasped time for reading LE, ' + dname + ': ' + str(elapsed_time))



# cfgs['photoC_TOT_zint_100m'].LE_path_load('photoC_TOT_zint_100m') 

# cfgs['photoC_TOT_zint_100m'].LE_ds = xr.open_mfdataset(
#     cfgs['photoC_TOT_zint_100m'].LE_file_list[0],
#     combine="nested",
#     concat_dim=[[*cfgs['photoC_TOT_zint_100m'].LE_ensembles], "time"],
#     preprocess=lambda ds: process_coords_2d(
#             'photoC_TOT_zint_100m', ds, sd, ed
#         ),
#     parallel=True,
# )

# # # rename ensemble dimension
# cfgs['photoC_TOT_zint_100m'].LE_ds = cfgs['photoC_TOT_zint_100m'].LE_ds.isel(concat_dim=0)
# new_time = cfgs['photoC_TOT_zint_100m'].LE_ds.time - np.array(
#     [datetime.timedelta(days=15)] * cfgs['photoC_TOT_zint_100m'].LE_ds.sizes["time"]
# )
# cfgs['photoC_TOT_zint_100m'].LE_ds = cfgs['photoC_TOT_zint_100m'].LE_ds.assign_coords(time=new_time)

# end_time = time.time()
# elapsed_time = end_time - start_time
# print('elasped time for reading LE, ' + dname + ': ' + str(elapsed_time))